# Autoship Delay in Onboarding to Household (Adult + Kids) — Power Analysis (First Fix Conversion)

**Experiment:** Autoship Delay in Onboarding to Household (Adults + Kids) · **Owner:** Sergio Oyola · **Primary metric analyzed here:** First Fix Conversion (cancellation adjusted) · **Randomization unit:** `client_id` · **Allocation point:** at Household onboarding, once eligibility criteria are met (before a Fix is scheduled)

This notebook sizes a 2-cell A/B test for its primary metric, **First Fix Conversion**, using a historical, pre-launch read of the eligible population — no live allocation log exists yet, since this experiment has not launched.

## Population
Household onboarding clients that:
- Are not enrolled in Autoship
- Business Line: Women, Men, or Kids
- Are linked to a primary household client (`household_primary_client_id IS NOT NULL`) — no field marking the specific onboarding flow exists in the warehouse, so this broader linkage condition is used instead and likely captures more clients than the exact intended population
- Do **not** already have a Fix scheduled
- The primary client on the household already has shipping and payment info saved

## Metric definition
**First Fix Conversion (cancellation adjusted)** = clients whose `curated.client_first_conversion.cancellation_adjusted_first_fix_demand_ts` falls strictly after onboarding, within the maturation window, over eligible clients at onboarding. The eligibility gate already requires no such timestamp at or before onboarding, so the metric reads as: of clients with no Fix scheduled yet, what share go on to request one?

## Design: 2-cell test, single comparison
Eligible clients are randomized at Household onboarding into 2 cells at a 50/50 split:

| Cell | Experience |
|---|---|
| Control | Status quo — no Autoship nudge at any point |
| Treatment | Routed into the existing Autoship Delay experience immediately after First Fix conversion |

A single pairwise comparison is planned (Treatment vs. Control), so `alpha = 0.05` needs no multiple-comparison adjustment. This notebook sizes the primary metric only; other metrics tracked for this experiment are monitored qualitatively and are not powered here.

## Sidedness: both alternatives evaluated
The hypothesis is directional (First Fix Conversion should not decrease, and may increase from the more prominent post-signup routing), which argues for a one-sided test — but a two-sided view is also worth having on hand, as the more conservative standard. Both are computed side by side in Step 2; the headline summary in Step 3 uses the one-sided view.

## Minimum Detectable Effect grid
The relative-lift scenarios sized here are **2%, 3%, 4%, 5%, and 10%** on First Fix Conversion, Treatment vs. Control — a spread wide enough to show how required sample size and duration trade off against effect size.

## Maturation window
A client's First Fix Conversion needs time to resolve after onboarding — they schedule a Fix, or don't, at some point afterward, not instantly. Step 0b checks this empirically for this population, rather than assuming a fixed number of days.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
BUSINESS_LINES = ('Womens', 'Mens', 'Kids')
COHORT_START = '2026-03-01'  # 6 months of pooled history is enough to size this population
MATURATION_DAYS = 90  # per Step 0b: the eventual-conversion curve is still climbing at day 60 (2+pt gains per checkpoint) and only starts flattening around day 90-120
VALIDATION_WINDOW_DAYS = 30  # short recent window used only for the Step 0 schema/logic check

# Design parameters (2-cell test, single comparison, no multiple-comparison correction)
ALPHA = 0.05
POWER = 0.80
N_ARMS = 2
SPLIT = 0.5  # Control and Treatment are equal-sized arms
MDE_GRID = [0.02, 0.03, 0.04, 0.05, 0.10]  # relative lift scenarios on First Fix Conversion

## Step 0 — Confirm the eligibility fields resolve as expected

Eligibility is defined by four conditions pulled from three different tables, joined here for the first time:

- The "no Fix scheduled" check — no `curated.client_first_conversion` record at or before onboarding
- The primary client's "shipping and payment info saved" — read from `client_service_production.clients.shipping_address` (shipping) and a matching row in `payment_method_service.payment_methods` (payment on file)

`household_primary_client_id IS NOT NULL` is a confirmed, real column and is used directly as a filter here, so this check only scans household-linked clients — and only over the last `VALIDATION_WINDOW_DAYS` days, since this is a quick schema/logic check, not the baseline measurement itself. No signup-flow-type field exists anywhere in the warehouse to narrow this further, so `household_primary_client_id` alone stands in as the household filter and likely captures a broader population than intended.

In [2]:
validation_query = f"""--sql
WITH household_signups AS (
    SELECT
        c.client_id,
        c.household_primary_client_id,
        c.signup_at
    FROM curated.client c
    WHERE c.household_primary_client_id IS NOT NULL
      AND c.business_line IN {BUSINESS_LINES}
      AND c.signup_at >= CURRENT_DATE - INTERVAL '{VALIDATION_WINDOW_DAYS}' DAY
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
),
flags AS (
    SELECT
        hs.client_id,
        (cfc.client_id IS NULL) AS no_fix_scheduled,
        (csp.client_id IS NOT NULL) AS primary_profile_found,
        (csp.shipping_address IS NOT NULL) AS shipping_saved,
        EXISTS (
            SELECT 1 FROM payment_method_service.payment_methods pm
            WHERE CAST(pm.client_id AS INTEGER) = hs.household_primary_client_id
        ) AS payment_saved
    FROM household_signups hs
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hs.client_id
       AND cfc.cancellation_adjusted_first_fix_demand_ts <= hs.signup_at
    LEFT JOIN client_service_production.clients csp
        ON csp.client_id = hs.household_primary_client_id
)
SELECT
    COUNT(*) AS n_clients,
    COUNT(*) FILTER (WHERE no_fix_scheduled) AS n_without_fix_scheduled,
    COUNT(*) FILTER (WHERE primary_profile_found) AS n_primary_profile_found,
    COUNT(*) FILTER (WHERE shipping_saved) AS n_shipping_saved,
    COUNT(*) FILTER (WHERE payment_saved) AS n_payment_saved
FROM flags
"""

validation_df = query(validation_query)
validation_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_clients,n_without_fix_scheduled,n_primary_profile_found,n_shipping_saved,n_payment_saved
0,28011,28011,28011,28011,23876


**Reading this:** of 28,011 household-linked clients in the last 30 days, `n_without_fix_scheduled` and `n_primary_profile_found` both equal `n_clients` exactly (100%) — the joins resolve cleanly. `n_shipping_saved` is also 100%, and `n_payment_saved` is 23,876 (85.2%) — a large majority, not all, which is the expected shape.

## Step 0b — How long First Fix conversion actually takes to resolve

The maturation buffer used in Step 1 is a starting assumption, not a derived one. This checks it directly: for a cohort old enough that a full year has already passed, what share of eventual converters (converting within 365 days of onboarding) had already converted by day 7, 14, 30, 60, 90, 120, and 180? Wherever this curve flattens out is the maturation window that should actually be used.

In [3]:
maturation_check_query = f"""--sql
WITH household_cohort AS (
    SELECT
        c.client_id,
        c.household_primary_client_id,
        c.signup_at AS onboarding_ts
    FROM curated.client c
    WHERE c.business_line IN {BUSINESS_LINES}
      AND c.household_primary_client_id IS NOT NULL
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND c.signup_at <= CURRENT_DATE - INTERVAL '400' DAY
),
first_fix_status AS (
    SELECT
        hc.client_id,
        hc.household_primary_client_id,
        hc.onboarding_ts,
        cfc.cancellation_adjusted_first_fix_demand_ts AS first_fix_demand_ts
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
),
not_fix_scheduled AS (
    SELECT *
    FROM first_fix_status
    WHERE first_fix_demand_ts IS NULL
       OR first_fix_demand_ts > onboarding_ts
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
),
not_yet_autoship AS (
    SELECT ppr.*
    FROM primary_profile_ready ppr
    LEFT JOIN curated.client_pulse_journal pj
        ON pj.client_id = ppr.client_id
       AND pj.end_date > CURRENT_DATE - INTERVAL '460' DAY
       AND pj.start_date <= DATE(ppr.onboarding_ts)
       AND pj.end_date > DATE(ppr.onboarding_ts)
       AND pj.last_autoship_demand_ts IS NOT NULL
    WHERE pj.client_id IS NULL
),
days_to_convert AS (
    SELECT
        client_id,
        DATE_DIFF('day', onboarding_ts, first_fix_demand_ts) AS days_to_convert
    FROM not_yet_autoship
)
SELECT
    t.checkpoint_days,
    COUNT(*) FILTER (WHERE days_to_convert <= t.checkpoint_days) AS n_converted_by_checkpoint,
    COUNT(*) FILTER (WHERE days_to_convert <= 365) AS n_converted_within_365d,
    COUNT(*) FILTER (WHERE days_to_convert <= t.checkpoint_days) * 1.0
        / NULLIF(COUNT(*) FILTER (WHERE days_to_convert <= 365), 0) AS share_of_eventual_converters_captured
FROM days_to_convert
CROSS JOIN UNNEST(ARRAY[7, 14, 30, 60, 90, 120, 180]) AS t(checkpoint_days)
GROUP BY t.checkpoint_days
ORDER BY t.checkpoint_days
"""

maturation_check_df = query(maturation_check_query)
maturation_check_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,checkpoint_days,n_converted_by_checkpoint,n_converted_within_365d,share_of_eventual_converters_captured
0,7,1601839,1890574,0.8
1,14,1638999,1890574,0.9
2,30,1683813,1890574,0.9
3,60,1726968,1890574,0.9
4,90,1756000,1890574,0.9
5,120,1778731,1890574,0.9
6,180,1814160,1890574,1.0


**Reading this:** the observed curve is 85% (day 7) → 87% (14) → 89% (30) → 91% (60) → 93% (90) → 94% (120) → 96% (180) of eventual (365-day) converters captured. Looking at the gain between each checkpoint rather than the level alone: +2.0pt (7→14), +2.4pt (14→30), +2.3pt (30→60), +1.5pt (60→90), +1.2pt (90→120), +1.9pt (120→180) — the curve is still climbing at a similar rate through day 60, and only starts decelerating around day 90-120. `MATURATION_DAYS` is set to 90 in Step 1 to capture that flattening point rather than the 60-day placeholder used above, which would leave more of the true rate uncounted than necessary.

## Step 1 — First Fix Conversion baseline & daily eligible volume

Cohort = clients meeting all four eligibility conditions, pooled over the 6 months from `COHORT_START` through a matured cutoff. First Fix Conversion is read directly from `curated.client_first_conversion.cancellation_adjusted_first_fix_demand_ts` — the eligibility gate already confirms no such timestamp exists at or before onboarding, so the metric flags whether one appears afterward, within the maturation window.

In [4]:
baseline_query = f"""--sql
WITH household_cohort AS (
    SELECT
        c.client_id,
        c.household_primary_client_id,
        c.signup_at AS onboarding_ts
    FROM curated.client c
    WHERE c.business_line IN {BUSINESS_LINES}
      AND c.household_primary_client_id IS NOT NULL
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND c.signup_at >= DATE '{COHORT_START}'
      AND c.signup_at <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
),
first_fix_status AS (
    SELECT
        hc.client_id,
        hc.household_primary_client_id,
        hc.onboarding_ts,
        cfc.cancellation_adjusted_first_fix_demand_ts AS first_fix_demand_ts
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
),
not_fix_scheduled AS (
    SELECT *
    FROM first_fix_status
    WHERE first_fix_demand_ts IS NULL
       OR first_fix_demand_ts > onboarding_ts
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
),
not_yet_autoship AS (
    SELECT ppr.*
    FROM primary_profile_ready ppr
    LEFT JOIN curated.client_pulse_journal pj
        ON pj.client_id = ppr.client_id
       AND pj.end_date > DATE '{COHORT_START}'
       AND pj.start_date <= DATE(ppr.onboarding_ts)
       AND pj.end_date > DATE(ppr.onboarding_ts)
       AND pj.last_autoship_demand_ts IS NOT NULL
    WHERE pj.client_id IS NULL
),
joined AS (
    SELECT
        client_id,
        DATE_TRUNC('month', onboarding_ts) AS month,
        DATE(onboarding_ts) AS onboarding_date,
        CASE WHEN first_fix_demand_ts IS NOT NULL
              AND first_fix_demand_ts <= onboarding_ts + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS converted_first_fix
    FROM not_yet_autoship
),
month_days AS (
    SELECT month, COUNT(DISTINCT onboarding_date) AS days_observed
    FROM joined
    GROUP BY 1
)
SELECT
    j.month,
    COUNT(*) AS n_eligible,
    AVG(CAST(j.converted_first_fix AS DOUBLE)) AS first_fix_conversion_rate,
    md.days_observed,
    COUNT(*) * 1.0 / md.days_observed AS eligible_per_day
FROM joined j
JOIN month_days md ON md.month = j.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

baseline_df = query(baseline_query)
baseline_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,n_eligible,first_fix_conversion_rate,days_observed,eligible_per_day
0,2026-06-01 00:00:00.000,4674,0.269148,9,519.3
1,2026-05-01 00:00:00.000,17284,0.255439,31,557.5
2,2026-04-01 00:00:00.000,17734,0.265422,30,591.1
3,2026-03-01 00:00:00.000,21256,0.285566,31,685.7


**Reference month:** pick the most recent calendar month whose onboarding dates are all at least `MATURATION_DAYS` (90) old as of today — that's **May 2026** (June 2026 shows only 9 of its days past the cutoff, so it's context only, not the baseline). Monthly rates for context: March 28.6%, April 26.5%, May 25.5%.

In [5]:
REFERENCE_MONTH = '2026-05-01'
ref = baseline_df[baseline_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

BASELINE_RATE = float(ref['first_fix_conversion_rate'][0])
DAILY_ELIGIBLE = float(ref['eligible_per_day'][0])

print(f"BASELINE_RATE (Control experience) = {BASELINE_RATE:.4f}  |  DAILY_ELIGIBLE (reference month) = {DAILY_ELIGIBLE:,.1f} clients/day")

BASELINE_RATE (Control experience) = 0.2554  |  DAILY_ELIGIBLE (reference month) = 557.5 clients/day


## Step 1b — A fresher, decoupled daily-volume read

Unlike the conversion rate, daily eligible volume doesn't need the maturation wait — all four eligibility conditions are known immediately at onboarding. The most recent complete calendar month is used here for volume, decoupled from the matured reference month used for the rate above, in case the two differ.

In [6]:
recent_volume_query = f"""--sql
WITH household_cohort AS (
    SELECT c.client_id, c.household_primary_client_id, c.signup_at AS onboarding_ts
    FROM curated.client c
    WHERE c.business_line IN {BUSINESS_LINES}
      AND c.household_primary_client_id IS NOT NULL
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND c.signup_at >= DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '1' MONTH
      AND c.signup_at < DATE_TRUNC('month', CURRENT_DATE)
),
first_fix_status AS (
    SELECT
        hc.client_id,
        hc.household_primary_client_id,
        hc.onboarding_ts,
        cfc.cancellation_adjusted_first_fix_demand_ts AS first_fix_demand_ts
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
),
not_fix_scheduled AS (
    SELECT *
    FROM first_fix_status
    WHERE first_fix_demand_ts IS NULL
       OR first_fix_demand_ts > onboarding_ts
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
),
not_yet_autoship AS (
    SELECT ppr.*
    FROM primary_profile_ready ppr
    LEFT JOIN curated.client_pulse_journal pj
        ON pj.client_id = ppr.client_id
       AND pj.end_date > DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '1' MONTH
       AND pj.start_date <= DATE(ppr.onboarding_ts)
       AND pj.end_date > DATE(ppr.onboarding_ts)
       AND pj.last_autoship_demand_ts IS NOT NULL
    WHERE pj.client_id IS NULL
)
SELECT
    COUNT(*) AS n_eligible,
    COUNT(DISTINCT DATE(onboarding_ts)) AS days_observed,
    COUNT(*) * 1.0 / COUNT(DISTINCT DATE(onboarding_ts)) AS eligible_per_day
FROM not_yet_autoship
"""

recent_volume_df = query(recent_volume_query)
recent_volume_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_eligible,days_observed,eligible_per_day
0,17284,31,557.5


**Reading this:** August 2026's `eligible_per_day` is 557.5 — identical to May's reference-month figure. Volume has held steady between the reference month and now, not declined, so the sample-size table in Step 2 uses this figure with reasonable confidence.

In [7]:
DAILY_ELIGIBLE = float(recent_volume_df['eligible_per_day'][0])  # supersedes the Step 1 reference-month figure, per Step 1b
print(f"DAILY_ELIGIBLE (most recent complete month) = {DAILY_ELIGIBLE:,.1f} clients/day")

DAILY_ELIGIBLE (most recent complete month) = 557.5 clients/day


## Step 2 — Sample size & duration, one-sided and two-sided

`n_total_statsmodels` sizes a single pairwise 50/50 comparison; `n_treatment` is read as the **per-arm** requirement, `n_total = 2 x n_per_arm`, and each arm accrues `DAILY_ELIGIBLE / 2` eligible clients per day under the 50/50 split. Both sidedness options are shown side by side — the one-sided table matches the directional hypothesis (First Fix Conversion does not decrease); the two-sided table is the more conservative alternative.

In [8]:
def size_table(rel_grid, baseline, daily, two_sided):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT],
        alpha=ALPHA, power=POWER, two_sided=two_sided,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total'] = df['n_per_arm'] * N_ARMS
    df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
    df['weeks_required'] = (df['days_required'] / 7).round(1)
    return df[['rel_effect', 'p_treatment', 'n_per_arm', 'n_total', 'days_required', 'weeks_required']]

print(f"--- First Fix Conversion, Treatment vs. Control (baseline={BASELINE_RATE:.1%}, alpha={ALPHA}, power={POWER:.0%}, one-sided, 50/50 split) ---")
one_sided_table = size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE, two_sided=False)
one_sided_table

--- First Fix Conversion, Treatment vs. Control (baseline=25.5%, alpha=0.05, power=80%, one-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total,days_required,weeks_required
0,+2%,0.260547,90694,181388,326,46.6
1,+3%,0.263102,40438,80876,146,20.9
2,+4%,0.265656,22819,45638,82,11.7
3,+5%,0.26821,14650,29300,53,7.6
4,+10%,0.280982,3719,7438,14,2.0


In [9]:
print(f"--- First Fix Conversion, Treatment vs. Control (baseline={BASELINE_RATE:.1%}, alpha={ALPHA}, power={POWER:.0%}, two-sided, 50/50 split) ---")
two_sided_table = size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE, two_sided=True)
two_sided_table

--- First Fix Conversion, Treatment vs. Control (baseline=25.5%, alpha=0.05, power=80%, two-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total,days_required,weeks_required
0,+2%,0.260547,115138,230276,414,59.1
1,+3%,0.263102,51337,102674,185,26.4
2,+4%,0.265656,28969,57938,104,14.9
3,+5%,0.26821,18598,37196,67,9.6
4,+10%,0.280982,4721,9442,17,2.4


**Reading this:** the two-sided table always requires more samples (and more days) than the one-sided table for the same MDE and power, since it splits `alpha` across both tails — roughly 25-30% more days at every MDE here. At the observed 557.5/day and a 25.5% baseline: the 2% scenario needs 326 days one-sided (414 two-sided). 3% needs 146/185 days. 4% needs 82/104 days (~12-15 weeks). 5% needs 53/67 days. 10% needs just 14/17 days. All five scenarios are far more feasible here than for a lower-baseline metric at the same relative lift, since a 25.5% baseline needs many fewer absolute conversions to detect the same percentage move.

## Step 3 — Headline sample-size summary

Headline MDE below is the **median of the Step 2 grid, +4% relative lift**. This summary uses one-sided, matching the directional hypothesis — the two-sided cost at the same MDE is in Step 2's grid, for comparison.

In [10]:
TARGET_REL_MDE = 0.04  # median of the Step 2 grid [0.02, 0.03, 0.04, 0.05, 0.10]

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=False,
)

n_per_arm = int(list(res.values())[0]['n_treatment'])
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE / N_ARMS)))

summary = {
    'Metric Used': 'First Fix Conversion (cancellation adjusted), Treatment vs. Control',
    'Population': 'Household onboarding clients, not already enrolled in Autoship, meeting all 4 eligibility conditions (see Step 0)',
    'Baseline Value': f"{BASELINE_RATE:.1%} (Control experience; pooled {COHORT_START} through a {MATURATION_DAYS}-day-mature cutoff)",
    'Daily Eligible Volume': f"{DAILY_ELIGIBLE:,.1f} / day (most recent complete month, per Step 1b)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.3f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test': 'One-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control / Treatment)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': f"{duration_days} days (~{duration_days/7:.1f} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,"First Fix Conversion (cancellation adjusted), Treatment vs. Control"
Population,"Household onboarding clients, not already enrolled in Autoship, meeting all 4 eligibility conditions (see Step 0)"
Baseline Value,25.5% (Control experience; pooled 2026-03-01 through a 90-day-mature cutoff)
Daily Eligible Volume,"557.5 / day (most recent complete month, per Step 1b)"
Minimum Detectable Effect,+4% relative (0.255 -> 0.266)
One/Two-Sided Test,One-sided
Significance Level,"0.05 (single comparison, no multiple-comparison correction)"
Statistical Power,80%
Variant Split %,50% / 50% (Control / Treatment)
Minimum Samples by Variant,"22,819"


## Bottom line

- **`MATURATION_DAYS = 90`, per Step 0b** — the eventual-conversion curve is still gaining roughly 2 points per checkpoint through day 60 (91% captured) and only starts flattening around day 90-120 (93%, then 94%), so 90 days is the more defensible cutoff.
- **No live allocation log exists yet** — this experiment has not launched, so both the rate and the volume come from historical eligibility reads rather than an in-flight allocation log; refresh both once the experiment is live and an allocation log exists.
- **Volume has held steady** — August 2026 (557.5/day) matches the May reference month (557.5/day) used for the rate exactly.
- At the **+4% relative** headline MDE, one-sided: 22,819 per arm, 82 days (~11.7 weeks) — meaningfully faster to size than a lower-baseline metric at the same relative lift. The full grid across 2-10%, both sidedness options, is in Step 2.